# থিসিস পাইপলাইন: সব ধাপ এক নোটবুকে
**Cross-network intrusion detection on NetFlow v3** · রিপো: `mirmumin/netflow-crossdataset-ids`

এই নোটবুক উপর থেকে নিচে চালালে থিসিসের সব ফলাফল আবার তৈরি হয়। প্রতিটা ধাপের আগে বাংলায় লেখা আছে **কী করে, কেন করে, কত সময় লাগে, আর ফলাফল কোথায় যায়**। কোডের ভেতরের কমেন্ট ইংরেজিতে, কারণ পেপার আর মক ডিফেন্সে কোড ইংরেজিতেই দেখাতে হবে।

| ধাপ | কাজ | Runtime | আনুমানিক সময় |
|---|---|---|---|
| 0 | সেটআপ: Drive, কোড, প্যাকেজ | যেকোনো | ২ মিনিট |
| 1 | ডাটা প্রস্তুত (সাবস্যাম্পল) | CPU | প্রথমবার ২০–৩০ মিনিট, পরে ১ মিনিট |
| 2 | ডাটা বিশ্লেষণ (EDA), ঐচ্ছিক | CPU | ২ মিনিট |
| 3 | বেসলাইন: RF, XGBoost, SVM, LR, KNN, Ensemble | CPU | ১–১.৫ ঘণ্টা |
| 4 | ফিউশন (অল্প টার্গেট লেবেল) | **GPU** | ২০–৩০ মিনিট |
| 5 | Benign-only টার্গেট (প্রস্তাবিত Fusion-B) | **GPU** | ১৫–২৫ মিনিট |
| 6 | Word রিপোর্ট ও টেবিল দেখা | যেকোনো | ১ মিনিট |

**সব ফলাফল সরাসরি Google Drive-এ** (`MyDrive/thesis_data/results/`) সেভ হয়। তাই Colab মাঝপথে বন্ধ হলেও যা হয়েছে তা হারায় না। একই সেল আবার চালালে যেখানে থেমেছিল সেখান থেকে চলবে।

**শুরুর আগে:** Runtime → Change runtime type → **T4 GPU** দিয়ে নিন। ধাপ ৪–৫-এ GPU লাগবে, বাকি ধাপ GPU-তেও চলে।

## ধাপ 0 · সেটআপ
**কী করে:** Google Drive যুক্ত করে, GitHub থেকে কোড আনে (রিপো Private, তাই Colab Secrets-এর `GH_TOKEN` লাগে), আর দরকারি প্যাকেজ ইনস্টল করে।

**আগে একবার করতে হবে:** বাম পাশের 🔑 আইকনে `GH_TOKEN` নামে GitHub টোকেন রাখুন, আর "Notebook access" অন করুন।

Drive-এর অনুমতি চাইলে "Select all" দিন, নইলে মাউন্ট হবে না।

In [ ]:
# --- Step 0: setup -------------------------------------------------------------
import os, subprocess
from google.colab import drive, userdata

# 1) Connect Google Drive (data and results live there permanently)
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/thesis_data'      # folder with the 4 zips / processed data
RES   = f'{DRIVE}/results'                        # all result files are written here
os.makedirs(RES, exist_ok=True)

# 2) Get the code. The repo is private, so we use the read-only token saved in Colab Secrets.
REPO = '/content/netflow-crossdataset-ids'
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    r = subprocess.run(['git', 'clone', f'https://{token}@github.com/mirmumin/netflow-crossdataset-ids.git', REPO],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***'))   # never print the token
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'])     # already cloned: just update
%cd /content/netflow-crossdataset-ids

# 3) Packages. Colab already has pandas, scikit-learn, xgboost; we add the rest.
!pip install -q tabpfn==9.2.0 python-docx pyyaml pyarrow

# 4) Show whether a GPU is available (needed for steps 4 and 5)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU only)')

## ধাপ 1 · ডাটা প্রস্তুত
**কী করে:** ৪টা NetFlow v3 ডাটাসেট (মোট প্রায় ১৩ GB) থেকে প্রতিটার প্রায় ৫ লাখ ফ্লো নেয়। প্রতিটা অ্যাটাক ক্লাস থেকে আনুপাতিক হারে নেওয়া হয়, যাকে বলে **stratified subsample**। ছোট ক্লাসগুলো পুরো রাখা হয়, আর অন্তত ১ লাখ বেনিন ফ্লো রাখা হয়।

**কেন:** পুরো ডাটা ফ্রি Colab-এর মেমরিতে আঁটে না। ৫ লাখ ফ্লো-ই যথেষ্ট বড় আর প্রতিনিধিত্বমূলক।

**কোথায় যায়:** `thesis_data/processed/*.parquet`। একবার তৈরি হলে পরের বার এই সেল শুধু কপি করে, ১ মিনিটে শেষ হয়।

কোড: `scripts/prepare_data.py`, `src/data.py`

In [ ]:
# --- Step 1: data preparation ---------------------------------------------------
import glob, zipfile, shutil
PROC_DRIVE = f'{DRIVE}/processed'
os.makedirs('data/processed', exist_ok=True)

if len(glob.glob(f'{PROC_DRIVE}/*.parquet')) == 4:
    # Already done in an earlier session: just copy the 4 small parquet files
    !cp {PROC_DRIVE}/*.parquet {PROC_DRIVE}/*.csv {PROC_DRIVE}/*.json data/processed/ 2>/dev/null
    print('Processed data copied from Drive.')
else:
    # First time: unzip the 4 UQ zip files from Drive into data/raw/
    os.makedirs('data/raw', exist_ok=True)
    for z in glob.glob(f'{DRIVE}/*.zip'):
        name = os.path.splitext(os.path.basename(z))[0]
        print('unzipping', name)
        zipfile.ZipFile(z).extractall(f'data/raw/{name}')
    # Stratified subsample (~500k flows per dataset) -> data/processed/*.parquet
    !python scripts/prepare_data.py
    os.makedirs(PROC_DRIVE, exist_ok=True)
    !cp data/processed/* {PROC_DRIVE}/

# Class balance of each subsample (rows = attack classes)
import pandas as pd
pd.read_csv('data/processed/class_summary.csv')

## ধাপ 2 · ডাটা বিশ্লেষণ (EDA), ঐচ্ছিক
**কী করে:** প্রতিটা ফিচারের মান এক নেটওয়ার্ক থেকে আরেক নেটওয়ার্কে কতটা আলাদা, তা মাপে (Kolmogorov–Smirnov স্ট্যাটিস্টিক: ০ মানে একই, ১ মানে সম্পূর্ণ আলাদা)। তারপর একটা হিটম্যাপ আঁকে।

**কেন:** এই ছবিটা পেপারে দেখাবে কেন এক নেটওয়ার্কে শেখা মডেল অন্য নেটওয়ার্কে ব্যর্থ হয়। এটাই থিসিসের মোটিভেশন।

**কোথায় যায়:** `results/figures/ks_shift_heatmap.png`, `attack_rate.png`

কোড: `notebooks/01_eda.ipynb`

In [ ]:
# --- Step 2 (optional): run the EDA notebook headless and keep its figures --------
!jupyter nbconvert --to notebook --execute notebooks/01_eda.ipynb --output 01_eda_run.ipynb
os.makedirs(f'{RES}/figures', exist_ok=True)
!cp results/figures/*.png {RES}/figures/ && cp results/ks_shift_benign.csv {RES}/
from IPython.display import Image, display
display(Image(f'{RES}/figures/ks_shift_heatmap.png', width=600))

## ধাপ 3 · বেসলাইন মডেল: ইন-ডাটাসেট ও ক্রস-ডাটাসেট
**কী করে:** ৫টা প্রচলিত মডেল (RF, XGBoost, SVM, LR, KNN) আর তাদের সমন্বিত **Ensemble** (৫টার অ্যাটাক-সম্ভাবনার গড়) ট্রেইন করে। প্রতিটা মডেল একটা নেটওয়ার্কে ট্রেইন হয়, তারপর ৪টা নেটওয়ার্কের টেস্ট অংশেই টেস্ট হয়।
- একই নেটওয়ার্কে টেস্ট হলে সেটা **ইন-ডাটাসেট** ফলাফল, এটাই স্যারের টেবিল।
- অন্য নেটওয়ার্কে টেস্ট হলে সেটা **ক্রস-ডাটাসেট** ফলাফল (৪×৪ = ১২টা জোড়া)। এখানেই সমস্যাটা ধরা পড়ে।

**ফলাফল (seed 42):** ইন-ডাটাসেটে F1 ৮৪–১০০%, কিন্তু ক্রস-ডাটাসেটে গড় F1 মাত্র ১৮–৩৩%।

**সময়:** ১–১.৫ ঘণ্টা। শেষ হওয়া অংশ আর চলে না, তাই মাঝপথে বন্ধ হলে আবার চালালেই হবে। চূড়ান্ত পেপারের জন্য `--all-seeds` যোগ করুন (৫টা seed, ৫ গুণ সময়)।

কোড: `scripts/run_baselines.py`, `src/models.py`, `src/evaluate.py`

In [ ]:
# --- Step 3: classical baselines (in-dataset + 4x4 cross-dataset) ----------------
# Results are appended to {RES}/baselines_raw.csv as each model finishes.
!python scripts/run_baselines.py --feature-sets v3 --results-dir {RES}

# Turn the raw rows into paper tables (supervisor's table = table_models.md)
!python scripts/summarise_baselines.py {RES}

## ধাপ 4 · ফিউশন: টার্গেটে অল্প লেবেল (k = 100 / 1000)
**পরিস্থিতি:** নতুন (টার্গেট) নেটওয়ার্কে মাত্র k-টা ফ্লো লেবেল করা আছে, বেনিন আর অ্যাটাক দুটোই। বাকি ৩টা (সোর্স) নেটওয়ার্কে প্রচুর লেবেল আছে।

**তুলনা হয় যাদের মধ্যে:**
- `XGB-src`: শুধু সোর্সে শেখা XGBoost, টার্গেটের কোনো লেবেল ছাড়া (zero-shot)
- `XGB-k`, `TabPFN-k`: শুধু টার্গেটের k লেবেলে শেখা মডেল
- `Fusion-k`: সোর্স XGBoost-এর স্কোর অতিরিক্ত ফিচার হিসেবে যোগ করে TabPFN-কে টার্গেটের k লেবেল দিয়ে শেখানো

**ফলাফল (seed 42):** Fusion ≈ TabPFN, কোনো বাড়তি লাভ নেই। মাত্র ১০০টা টার্গেট লেবেলেই F1 ৮৪–৯৯%। মানে, এই পরিস্থিতিতে সোর্সের জ্ঞান কাজে আসে না। Rayamajhi (2026)-ও একই কথা বলেছেন। এ কারণেই আমরা ধাপ ৫-এর দিকে গেছি।

কোড: `scripts/run_fusion.py`, `src/fusion.py`

In [ ]:
# --- Step 4: few-shot fusion (needs GPU for TabPFN) ------------------------------
!python scripts/run_fusion.py --results-dir {RES}

## ধাপ 5 · Benign-only টার্গেট (প্রস্তাবিত Fusion-B) ⬅ **এখন এই ধাপটা বাকি**
**পরিস্থিতি (বাস্তবসম্মত):** নতুন নেটওয়ার্কের অপারেটর নিজের **স্বাভাবিক ট্রাফিক** সহজেই রেকর্ড করতে পারেন। কিন্তু **অ্যাটাকের লেবেল** তাঁর কাছে নেই। তাই অ্যাটাক চেনার জ্ঞান কেবল অন্য নেটওয়ার্ক থেকেই আসতে পারে। এই পরিস্থিতিতে সোর্স জ্ঞান অপরিহার্য।

**তুলনা হয় যাদের মধ্যে:**
- `XGB-src`: শুধু সোর্সে শেখা XGBoost
- `IForest-k`: Isolation Forest, যা শুধু টার্গেটের k বেনিন দেখে "অস্বাভাবিক" ফ্লো ধরে (সোর্স ছাড়া)
- `XGB-pool-k`: সোর্স ডাটা আর টার্গেটের বেনিন একসাথে মিলিয়ে XGBoost
- `Fusion-B-k` (**আমাদের প্রস্তাব**): TabPFN দেখে টার্গেটের k বেনিন আর সোর্সের অ্যাটাক উদাহরণ। সাথে দুটো অতিরিক্ত সংকেত পায়: সোর্স XGBoost-এর স্কোর, আর টার্গেটের অস্বাভাবিকতা স্কোর।

**সময়:** GPU-তে ১৫–২৫ মিনিট।

কোড: `scripts/run_benign_only.py`

In [ ]:
# --- Step 5: benign-only target setting, proposed Fusion-B (needs GPU) ----------
!python scripts/run_benign_only.py --results-dir {RES}

## ধাপ 6 · Word রিপোর্ট ও টেবিল
**কী করে:** এ পর্যন্ত যত ফলাফল আছে, সব মিলিয়ে `report.docx` বানায়: Setup, স্যারের টেবিল, ক্রস-ডাটাসেট, ফিউশন, Benign-only। নিচের সেল নোটবুকের ভেতরেই মূল টেবিলগুলো দেখায়।

**কোথায় যায়:** `MyDrive/thesis_data/results/report.docx`। Word বা Google Docs-এ খুলে স্যারকে দেখানো যায়।

কোড: `scripts/make_report.py`

In [ ]:
# --- Step 6: Word report + show the key tables here -------------------------------
!python scripts/make_report.py {RES}

from IPython.display import Markdown, display
if os.path.exists(f'{RES}/table_models.md'):                 # supervisor's table (in-dataset)
    display(Markdown(open(f'{RES}/table_models.md').read()))
for name in ['table_cross_summary.csv', 'fusion_summary.csv', 'benign_only_v2_summary.csv']:
    path = f'{RES}/{name}'
    if os.path.exists(path):
        display(Markdown(f'**{name}**'))
        display(pd.read_csv(path))

---
### কোন ফাইলে কী আছে (মক ডিফেন্সের জন্য)
| ফাইল | কাজ |
|---|---|
| `configs/default.yaml` | সব সেটিং এক জায়গায়: seed, সাবস্যাম্পল সাইজ, মডেলের তালিকা |
| `src/data.py` | বড় CSV থেকে stratified subsample |
| `src/features.py` | কোন ফিচার ব্যবহার হবে; IP/পোর্ট/টাইমস্ট্যাম্প বাদ; infinity ঠিক করা ও log(1+x) |
| `src/splits.py` | ৮০/২০ train/test ভাগ |
| `src/models.py` | RF, XGBoost, SVM, LR, KNN আর Ensemble |
| `src/evaluate.py` | Accuracy, Precision, Recall, F1, Specificity ইত্যাদি |
| `src/fusion.py` | TabPFN লোড করা, স্কোর রূপান্তর |
| `scripts/*.py` | প্রতিটা ধাপের স্ক্রিপ্ট (উপরের ধাপ ১–৬) |
| `tests/test_data.py` | নকল ছোট ডাটায় পুরো পাইপলাইন ঠিক চলে কিনা পরীক্ষা |